title: "Individual Planning Report"
author: Yingnian Le
output: html_document

# (1) Data Description
The dataset consists of two related files: "players.csv" and "sessions.csv", collected from a Minecraft server operated by Frank Wood in lab at UBC.
players.csv:
| Variable | Type |	Description |
| experience |	Categorical	| Player experience level (e.g., “Amateur”, “Pro”, “Veteran”) |
| subscribe	| Logical |	Whether the player subscribed to the game-related newsletter (TRUE/FALSE) |
| hashedEmail |String |	Unique hashed identifier for each player |
| played_hours | Numeric |Total number of hours the player spent in the game |
| name | String	| Player’s name |
| gender | Categorical | Player gender |
| Age | Numeric	| Player age |
session.csv:
| Varaible | Type | Description |
| hashedEmail | String | Links session to a player |
| start_time | Datetime	| Session start (DD/MM/YYYY HH:MM) |
| end_time | Datetime | Session end (DD/MM/YYYY HH:MM) |
| original_start_time |	Numeric | Original timestamp form |
| original_end_time | Numeric | Original timestamp form |


Data issues: Possibly missing values in played_hours or Age.
Duplicates could exist if the same session was logged twice.
Time data may need to be converted into consistent time zone. Sampling bias might exist, only players on the research server are recorded.

# (2) Questions
Broad question:
What player characteristics and behaviours are most predictive of subscribing to a game-related newsletter, and how do these features differ between various player types?

Specific question: Can player experience level, playtime, gender, and age predict whether a player subscribes to the newsletter?

Response variable: subcribe (TRUE/FALSE)
Explanatory variables: experience, played-hours, Age, gender

## (3) Exploratory Data Analysis and Visualization

In [3]:
#Load packages
library(tidyverse)
library(tidymodels)
library(lubridate)

── Attaching packages ────────────────────────────────────── tidymodels 1.1.1 ──

✔ broom        1.0.6     ✔ rsample      1.2.1
✔ dials        1.3.0     ✔ tune         1.1.2
✔ infer        1.0.7     ✔ workflows    1.1.4
✔ modeldata    1.4.0     ✔ workflowsets 1.0.1
✔ parsnip      1.2.1     ✔ yardstick    1.3.1
✔ recipes      1.1.0     

── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() masks purrr::discard()
✖ dplyr::filter()   masks stats::filter()
✖ recipes::fixed()  masks stringr::fixed()
✖ dplyr::lag()      masks stats::lag()
✖ yardstick::spec() masks readr::spec()
✖ recipes::step()   masks stats::step()
• Use tidymodels_prefer() to resolve common conflicts.



In [4]:
# Load datasets
players <- read_csv("data/players.csv")
sessions <- read_csv("data/sessions.csv")

glimpse(players)
glimpse(sessions)

Rows: 196 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): experience, hashedEmail, name, gender
dbl (2): played_hours, Age
lgl (1): subscribe

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1535 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): hashedEmail, start_time, end_time
dbl (2): original_start_time, original_end_time

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 196
Columns: 7
$ experience   <chr> "Pro", "Veteran", "Veteran", "Amateur", "Regular", "Amate…
$ subscribe    <lgl> TRUE, TRUE, FALSE, TRUE, TRUE, TRUE, TRUE, FALSE, TRUE, T…
$ hashedEmail  <chr> "f6daba428a5e19a3d47574858c13550499be23603422e6a0ee9728f8…
$ played_hours <dbl> 30.3, 3.8, 0.0, 0.7, 0.1, 0.0, 0.0, 0.0, 0.1, 0.0, 1.6, 0…
$ name         <chr> "Morgan", "Christian", "Blake", "Flora", "Kylie", "Adrian…
$ gender       <chr> "Male", "Male", "Male", "Female", "Male", "Female", "Fema…
$ Age          <dbl> 9, 17, 17, 21, 21, 17, 19, 21, 47, 22, 23, 17, 25, 22, 17…
Rows: 1,535
Columns: 5
$ hashedEmail         <chr> "bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431d8a…
$ start_time          <chr> "30/06/2024 18:12", "17/06/2024 23:33", "25/07/202…
$ end_time            <chr> "30/06/2024 18:24", "17/06/2024 23:46", "25/07/202…
$ original_start_time <dbl> 1.71977e+12, 1.71867e+12, 1.72193e+12, 1.72188e+12…
$ original_end_time   <dbl> 1.71977e+12, 1.71867e+12, 1.72193e+12, 1.72188e+